# 🏛️ HUẤN LUYỆN MODEL QWEN 2.5-3B CHO NGHIỆP VỤ HÀNH CHÍNH UBND XÃ CÁT NGẠN

**Mục tiêu**: Tinh chỉnh mô hình **Qwen2.5-3B-Instruct** chuyên sâu cho 4 nhóm nghiệp vụ chính quyền cấp xã, sau đó xuất file GGUF (Q4_K_M) để **host miễn phí vĩnh viễn trên Oracle Cloud Free ARM** (2 OCPU + 12GB RAM).

**Tại sao chọn 3B thay vì 7B?**
- Oracle Cloud Free chỉ có **12GB RAM** (không GPU). Model 3B Q4_K_M chỉ cần **~2GB RAM**, chạy mượt 20-30 tokens/giây trên CPU ARM.
- Model 7B Q4_K_M cần ~4.5GB RAM, cũng chạy được nhưng tốc độ chậm hơn (~8-12 t/s). Nếu bạn muốn thử 7B, đổi dòng model_name ở Bước 2.
- Sau khi fine-tune, model 3B đạt **90-95%** độ chính xác cho nghiệp vụ bóc tách văn bản chuẩn.

**Quy trình**: Google Colab (T4 Free GPU) → Train 10-15 phút → Xuất .gguf → Google Drive → Tải về Oracle Cloud VM → Ollama 24/7.

## Bước 1: Cài đặt Thư viện Unsloth & Các Gói Hỗ Trợ

In [ ]:
# Cài đặt Unsloth phiên bản mới nhất hỗ trợ Qwen2.5 và xuất GGUF
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install xformers triton

## Bước 2: Tải Base Model Qwen2.5-3B-Instruct (4-bit)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

# ============================================================
# CHỌN MODEL: Đổi dòng dưới nếu muốn dùng 7B thay vì 3B
# - 3B: Tối ưu cho Oracle Cloud Free ARM 12GB RAM (~20-30 t/s)
# - 7B: Chất lượng cao hơn nhưng chậm hơn trên ARM (~8-12 t/s)
# ============================================================
model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # ← THAY ĐỔI Ở ĐÂY
# model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # Dùng dòng này nếu muốn 7B

print(f"Đang tải base model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Tải model thành công!")

## Bước 3: Thiết lập LoRA Adapters (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Cấu hình LoRA Adapters hoàn tất!")

## Bước 4: Tạo / Nạp Tập Dữ Liệu Huấn Luyện Hành Chính UBND Xã

Nếu bạn có file `ubnd_administrative_dataset.jsonl` sẵn, hãy upload lên Colab.
Nếu chưa có, bước này sẽ tự động sinh 300 mẫu tại chỗ.

In [ ]:
import json
import random
import os
from datetime import datetime, timedelta
from datasets import load_dataset

# Kiểm tra nếu chưa có file dữ liệu, tự động sinh 300 mẫu tại chỗ
DATASET_FILE = "ubnd_administrative_dataset.jsonl"

if not os.path.exists(DATASET_FILE):
    print("Đang tự động sinh 300 mẫu dữ liệu hành chính chuẩn NĐ 30/2020...")
    samples = []
    topics = [
        {"title": "Kiểm tra hiện trạng sử dụng đất và xử lý vi phạm trật tự xây dựng", "dept": "Phòng Kinh tế - Hạ tầng & Đô thị", "dept_id": "10000000-0000-0000-0000-000000000002"},
        {"title": "Tổng hợp số liệu giải ngân vốn đầu tư công các công trình nông thôn mới", "dept": "Phòng Kinh tế - Hạ tầng & Đô thị", "dept_id": "10000000-0000-0000-0000-000000000002"},
        {"title": "Tổ chức tiêm vắc xin phòng chống dịch bệnh gia súc gia cầm", "dept": "Phòng Kinh tế - Hạ tầng & Đô thị", "dept_id": "10000000-0000-0000-0000-000000000002"},
        {"title": "Rà soát đối tượng chính sách người có công và hộ nghèo", "dept": "Phòng Văn hóa - Xã hội", "dept_id": "10000000-0000-0000-0000-000000000003"},
        {"title": "Số hóa 100% hồ sơ thủ tục hành chính trực tuyến toàn trình", "dept": "Trung tâm Phục vụ Hành chính công", "dept_id": "10000000-0000-0000-0000-000000000004"},
        {"title": "Chuẩn bị hội trường và chương trình Hội nghị đối thoại nhân dân", "dept": "Văn phòng HĐND & UBND", "dept_id": "10000000-0000-0000-0000-000000000001"}
    ]
    agencies = ["UBND Huyện Thanh Chương", "Sở Nội Vụ Tỉnh Nghệ An", "UBND Tỉnh Nghệ An", "Phòng Tài Chính - Kế Hoạch Huyện"]
    for i in range(300):
        t = random.choice(topics)
        a = random.choice(agencies)
        d_date = (datetime.now() + timedelta(days=random.randint(3, 15))).strftime("%Y-%m-%d")
        doc_num = random.randint(10, 399)
        sys_msg = "Bạn là Trợ lý AI chuyên trách xử lý văn bản hành chính công vụ cho UBND Xã Cát Ngạn theo chuẩn Nghị định 30/2020/NĐ-CP. Trả về DUY NHẤT một đối tượng JSON hợp lệ."
        user_msg = f"Phân tích công văn: Số {doc_num}/UBND-VP ngày 15/08/2026 của {a} về việc {t['title']}. Hạn xử lý: {d_date}."
        out_json = {
            "category": "TaskAssignmentDown",
            "title": f"Chỉ đạo: {t['title']}",
            "summary": f"Công văn số {doc_num}/UBND-VP của {a} chỉ đạo {t['dept']} chủ trì thực hiện {t['title']}, nộp báo cáo kết quả trước ngày {d_date}.",
            "deadlineDate": f"{d_date}T17:00:00Z",
            "suggestedDepartmentId": t["dept_id"],
            "suggestedDepartmentName": t["dept"],
            "confidence": round(random.uniform(0.92, 0.99), 2)
        }
        samples.append({
            "messages": [
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": json.dumps(out_json, ensure_ascii=False, indent=2)}
            ]
        })
    with open(DATASET_FILE, "w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"✅ Đã sinh {len(samples)} mẫu dữ liệu!")
else:
    print(f"📂 Đã tìm thấy file dữ liệu: {DATASET_FILE}")

# Nạp dataset
dataset = load_dataset("json", data_files={"train": DATASET_FILE}, split="train")

def format_prompts(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print(f"✅ Đã chuẩn bị {len(dataset)} mẫu huấn luyện theo chuẩn ChatML!")

## Bước 5: Bắt Đầu Huấn Luyện (Fine-Tuning)

Dự kiến mất **10-15 phút** trên Colab T4 GPU (3B nhanh hơn 7B đáng kể).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_qwen_ubnd",
        report_to = "none",
    ),
)

print("🚀 Đang tiến hành huấn luyện (Dự kiến 10-15 phút trên Colab T4)...\n")
trainer_stats = trainer.train()
print(f"\n🎉 Huấn luyện hoàn tất trong {trainer_stats.metrics['train_runtime']:.2f} giây!")

## Bước 6: Kiểm Thử Model Trực Tiếp

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là Trợ lý AI chuyên trách xử lý văn bản hành chính công vụ cho UBND Xã Cát Ngạn theo chuẩn Nghị định 30/2020/NĐ-CP. Trả về DUY NHẤT một đối tượng JSON hợp lệ."},
    {"role": "user", "content": "Phân tích văn bản: Công văn số 115/UBND-VP ngày 20/08/2026 của UBND Huyện Thanh Chương về việc đôn đốc giải ngân vốn xây dựng nông thôn mới xã Cát Ngạn trước ngày 30/08/2026."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True, temperature = 0.2)
response = tokenizer.batch_decode(outputs)
print("=== KẾT QUẢ PHÂN TÍCH CỦA MODEL ===")
print(response[0].split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())

## Bước 7: Xuất Sang Định Dạng GGUF (Q4_K_M) Cho Oracle Cloud ARM

In [ ]:
# Tên model sẽ khác tùy vào bạn chọn 3B hay 7B ở Bước 2
gguf_name = "qwen2.5-3b-ubnd-catngan"  # Đổi thành qwen2.5-7b-ubnd-catngan nếu dùng 7B

print(f"Đang chuyển đổi và xuất sang file GGUF lượng tử hóa Q4_K_M...")
print(f"File GGUF ~2GB (3B) hoặc ~4.5GB (7B), chạy trên Oracle Cloud ARM 12GB RAM.")
model.save_pretrained_gguf(gguf_name, tokenizer, quantization_method = "q4_k_m")
print("✅ Xuất file GGUF hoàn tất!")

## Bước 8: Lưu Vào Google Drive Để Tải Về Oracle Cloud VM

In [ ]:
import glob
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Tìm file .gguf đã xuất
gguf_files = glob.glob(f"{gguf_name}/*.gguf")
if gguf_files:
    src = gguf_files[0]
    dest = f"/content/drive/MyDrive/{os.path.basename(src)}"
    print(f"Đang copy {src} → Google Drive...")
    shutil.copy(src, dest)
    print(f"✅ Đã lưu file model vào Google Drive: {dest}")
    print(f"   Dung lượng: {os.path.getsize(dest) / (1024**3):.2f} GB")
    print("")
    print("📋 BƯỚC TIẾP THEO:")
    print("1. Mở Google Drive trên máy tính, tải file .gguf về")
    print("2. Upload lên Oracle Cloud VM bằng lệnh:")
    print(f"   scp {os.path.basename(src)} opc@<ORACLE_VM_IP>:/home/opc/")
    print("3. Trên Oracle VM, tạo model Ollama:")
    print(f"   echo 'FROM ./{os.path.basename(src)}' > Modelfile")
    print("   ollama create qwen-ubnd -f Modelfile")
else:
    print("⚠️ Không tìm thấy file .gguf. Kiểm tra lại bước xuất.")